In [116]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from urllib.parse import urlparse
from tempfile import mkdtemp
from typing import Union
from enum import Enum
import numpy as np
import logging
import pickle
import shutil
import json
import os
import re

In [117]:
DATA_FILE = '/home/joao/my/ita/mestrado/2-clustering-phishing-kit/utils/data.json'

In [118]:
a = np.array(['0.12', '0.23', '0.34'], dtype=np.float32)

In [119]:
from typing import List
from itertools import combinations

def cross_cosine_similarity(a: List[np.ndarray], b: List[np.ndarray]) -> float:
    max_similarity = -1
    most_similar_vectors = []

    for vector_a in a:
        for vector_b in b:
            similarity = cosine_similarity(vector_a.reshape(1, -1), vector_b.reshape(1, -1))
            if similarity > max_similarity:
                max_similarity = similarity
                most_similar_vectors = [(vector_a, vector_b)]

    return most_similar_vectors, max_similarity

def import_data(file):
    with open(file, 'r') as f:
        data = json.load(f)

    for filehash, segments in data.items():
        for idx, info in segments.items():
            info['vector'] = np.array(info['vector'], dtype=np.float32)

    hashes = [filehash for filehash in data.keys()]
    segmented_data = [[info['vector'] for info in segment.values()] for segment in data.values()]
    flat_data = np.array([info['vector'] for segment in data.values() for info in segment.values()], dtype=np.float32)

    dm = cosine_similarity(flat_data, flat_data)
    np.fill_diagonal(dm, 0.0)  # Replace diagonal values with 0.0

    choosen_hashes = []
    choosen_segments = []

    num_samples = len(segmented_data)
    total_segments_count = 0
    for i in range(num_samples):
        num_segments = len(segmented_data[i])
        current_segments = flat_data[total_segments_count:total_segments_count+num_segments]
        total_segments_count += num_segments

        current_segments_distances = dm[total_segments_count:total_segments_count+num_segments, :]

        if current_segments_distances.shape[0] == 0:
            continue

        max_similarity_indices = np.unravel_index(np.argmax(current_segments_distances), current_segments_distances.shape)
        most_similar_vector_x = current_segments[max_similarity_indices[0]]

        choosen_hashes.append(hashes[i])
        choosen_segments.append(most_similar_vector_x)

    # segments = []
    # for segment in new_data:
    #     most_similar_vector = None
    #     max_similarity = -1
    #     for other_segment in new_data:
    #         if id(segment) == id(other_segment):
    #             continue

    #         most_similar_vectors, similarity = cross_cosine_similarity(segment, other_segment)

    #         if similarity > max_similarity:
    #             max_similarity = similarity
    #             most_similar_vector = most_similar_vectors[0]


    # return segments
    return np.array(choosen_segments, dtype=np.float32), np.array(choosen_hashes)

In [120]:
X, y = import_data(DATA_FILE)

In [121]:
with open('vectors2.tsv', 'w') as f:
    for i in range(X.shape[0]):
        f.write('\t'.join([str(x) for x in X[i]]) + '\n')

In [122]:
from sklearn.cluster import DBSCAN

# Create an instance of DBSCAN
dbscan = DBSCAN(eps=0.05, min_samples=1, metric='cosine')

# Fit the data to DBSCAN
dbscan.fit(X)

# Get the labels assigned by DBSCAN
labels = dbscan.labels_

# Print the labels
print(labels)

[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16   4
  17  18   3  17  19  20  21  22  23   3  24  11  25  26  27  28  29  30
  31  32  33  34  35  36  37  15  38  39  26  40  41  42  43  13  44  20
  45  46  47  48  44  49  50  32  51  52  53  54  55  56  57   5  54  13
  20  20  58  20  59  60  61  62  63  59  64  65  59   5  60  66   4  67
  68  33  69  44  70  71  72  73  60  74  63  75  76  71  77  78  79  80
  79  44  81  66  82  63  83  84  85  86  87  20  60  88  89   5  77   4
  90   4  32  79  91  92  13  93  94  95  96   5  17  65  44  97  51  98
  99  28  99  65 100  71 101 102 103 104 105 106  75 107 108 109  20 110
 111  13  61  56   5 112 113  56  15 114  20  56  99  21 115 116  11 117
  20 118 119 106 120 121 122 123 124 125 126 127 128  51 129 130 131 132
 133  20 134 135 136 137 138 139 100 140 141 142 143   5  20  11 144  15
 145 130 137  33 146 147  56 148 149  20 150 151 152 153  33 154 155 152
 156 157 158]


In [123]:
with open('metadata2.tsv', 'w') as f:
    f.write('hash\tlabel\n')
    for i in range(y.shape[0]):
        f.write(y[i] + '\t' + str(labels[i]) + '\n')